# Session 13 · Vertex training, registry, and serving

Queen City Trip Analytics, a fictional South End, Charlotte company, predicts each fleet vehicle's fuel economy so a fare quote can carry its fuel cost.

In [ ]:
source ~/dsba6190-live-demo-13/env.sh && echo "$ENDPOINT_ID" \
  || echo "NOT STAGED. Run prep.ipynb first. Do not run any other cell."

## Step 1 · The trainer, packaged for a prebuilt container

In [ ]:
tar -tzf qc_fuel_trainer-0.1.tar.gz

In [ ]:
tar -xzOf qc_fuel_trainer-0.1.tar.gz qc_fuel_trainer-0.1/trainer/train.py | sed -n '/plain list/,/^mae/p'

## Step 2 · Two training jobs, already finished

In [ ]:
for j in "$JOB_RIDGE" "$JOB_BOOSTED"; do
  gcloud ai custom-jobs describe "$j" --region "$REGION" --project "$PROJECT" \
    --format='value(displayName,state,createTime.date(%H:%M:%S),endTime.date(%H:%M:%S),jobSpec.workerPoolSpecs[0].machineSpec.machineType)'
done

## Step 3 · Test error in the log, artifacts in the bucket

In [ ]:
for j in "$JOB_RIDGE" "$JOB_BOOSTED"; do
  gcloud logging read "resource.type=ml_job AND resource.labels.job_id=${j##*/} AND test_mae_mpg" \
    --project "$PROJECT" --limit 1 --format='value(textPayload,jsonPayload.message)'
done

In [ ]:
gcloud storage ls -l "gs://$BUCKET/models/*/model/"

## Step 4 · One model in the registry

In [ ]:
gcloud ai models describe "$MODEL_ID" --region "$REGION" --project "$PROJECT" \
  --format="yaml(displayName,versionId,versionAliases,artifactUri,containerSpec.imageUri)"

## Step 5 · The endpoint, already deployed

In [ ]:
gcloud ai endpoints describe "$ENDPOINT_ID" --region "$REGION" --project "$PROJECT" \
  --format="yaml(displayName,deployedModels)"

## Step 6 · The request, in raw units

In [ ]:
cat > request.json <<'JSON'
{"instances": [[8, 350, 165, 3693, 11.5, 70, 1], [4, 97, 88, 2130, 14.5, 71, 3], [4, 140, 86, 2790, 15.6, 82, 1]]}
JSON
cat request.json

## Step 7 · Version 2, then submit the batch job on it

In [ ]:
gcloud ai models upload --region "$REGION" --project "$PROJECT" --display-name "qc-fuel-$SUFFIX" \
  --parent-model "projects/$PROJECT/locations/$REGION/models/$MODEL_ID" \
  --artifact-uri "gs://$BUCKET/models/boosted/model" --container-image-uri "$SERVE_IMAGE" \
  --version-aliases=boosted

In [ ]:
gcloud ai models list-version "$MODEL_ID" --region "$REGION" --project "$PROJECT" \
  --format="table(versionId,versionAliases.list(),versionCreateTime.date('%H:%M:%S'),artifactUri.basename())"

In [ ]:
cat > batch.json <<JSON
{"displayName": "qc-fuel-batch-$SUFFIX",
 "model": "projects/$PROJECT/locations/$REGION/models/$MODEL_ID@boosted",
 "inputConfig": {"instancesFormat": "jsonl", "gcsSource": {"uris": ["gs://$BUCKET/batch/input.jsonl"]}},
 "outputConfig": {"predictionsFormat": "jsonl", "gcsDestination": {"outputUriPrefix": "gs://$BUCKET/batch/out"}},
 "dedicatedResources": {"machineSpec": {"machineType": "n1-standard-2"}, "startingReplicaCount": 1, "maxReplicaCount": 1}}
JSON
API="https://$REGION-aiplatform.googleapis.com/v1/projects/$PROJECT/locations/$REGION"
date +%s > batch-start.txt
curl -s -X POST -H "Authorization: Bearer $(gcloud auth print-access-token)" \
  -H 'Content-Type: application/json' "$API/batchPredictionJobs" -d @batch.json \
  | python3 -c 'import json,sys; d=json.load(sys.stdin); print(d.get("name", d)); print(d.get("state", ""))' | tee batch-job.txt

## Step 8 · One online prediction, priced

In [ ]:
time gcloud ai endpoints predict "$ENDPOINT_ID" --region "$REGION" --project "$PROJECT" --json-request request.json

In [ ]:
gcloud ai endpoints predict "$ENDPOINT_ID" --region "$REGION" --project "$PROJECT" --json-request request.json --format=json 2>/dev/null \
  | python3 -c 'import json,sys; p=json.load(sys.stdin)["predictions"]; [print(f"vehicle {i+1}: {m:5.1f} mpg  fuel for 20 miles ${20/m*3.20:5.2f}") for i,m in enumerate(p)]'

## Step 9 · Two cost shapes, then collect the batch job

In [ ]:
API="https://$REGION-aiplatform.googleapis.com/v1/projects/$PROJECT/locations/$REGION"
BJ=$(grep -o 'batchPredictionJobs/[0-9]*' batch-job.txt | head -1); echo "$BJ"
until s=$(curl -s -H "Authorization: Bearer $(gcloud auth print-access-token)" "$API/$BJ" | python3 -c 'import json,sys; print(json.load(sys.stdin).get("state", ""))') \
      && [[ "$s" =~ SUCCEEDED|FAILED|CANCELLED ]]; do
  echo "$(date +%H:%M:%S)  $s"; sleep 30
done
echo "batch job $s after $(( $(date +%s) - $(cat batch-start.txt) )) s"

In [ ]:
gcloud storage cat "gs://$BUCKET/batch/out/*/prediction.results-*" | head -3
echo "$(gcloud storage cat "gs://$BUCKET/batch/out/*/prediction.results-*" | wc -l | tr -d ' ') predictions written"

## Step 10 · Undeploy the endpoint

In [ ]:
DM=$(gcloud ai endpoints describe "$ENDPOINT_ID" --region "$REGION" --project "$PROJECT" --format="value(deployedModels[0].id)"); echo "$DM"

In [ ]:
gcloud ai endpoints undeploy-model "${ENDPOINT_ID:?}" --deployed-model-id "${DM:?}" \
  --region "${REGION:?}" --project "${PROJECT:?}" --quiet

In [ ]:
gcloud ai endpoints describe "$ENDPOINT_ID" --region "$REGION" --project "$PROJECT" \
  --format="yaml(displayName,deployedModels)"

In [ ]:
gcloud ai endpoints predict "$ENDPOINT_ID" --region "$REGION" --project "$PROJECT" --json-request request.json

## Step 11 · Teardown, then verify

In [ ]:
gcloud ai endpoints delete "${ENDPOINT_ID:?}" --region "${REGION:?}" --project "${PROJECT:?}" --quiet
gcloud ai models delete "${MODEL_ID:?}" --region "${REGION:?}" --project "${PROJECT:?}" --quiet

In [ ]:
gcloud storage rm -r "gs://${BUCKET:?}" --project "${PROJECT:?}"

In [ ]:
gcloud ai endpoints list --region "$REGION" --project "$PROJECT" --filter="displayName~qc-fuel"
gcloud ai models list --region "$REGION" --project "$PROJECT" --filter="displayName~qc-fuel"